In [49]:
import torch
from torch.utils.data import DataLoader, random_split
import numpy as np

import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import sys
from pathlib import Path
GAN_CLASSES = [3, 5]
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.data.preprocessing import load_multiple_subjects
from src.data.H5_dataset import EEGH5Dataset
from src.data.class_dataset import EEGClassDataset
from src.utils.logging import get_logger
from src.utils.config import (
    LATENT_DIM,
    LR_GENERATOR,
    LR_DISCRIMINATOR,
    EPOCHS_JOINT,
    WINDOW_SIZE,
    WINDOW_STRIDE,
    BATCH_SIZE,
    NUM_CHANNELS,
    WANTED_CHANNELS,
    NUM_LAYERS_EMBEDDER,
    NUM_LAYERS_SUPERVISOR,
    NUM_LAYERS_GENERATOR,
    NUM_LAYERS_DISCRIMINATOR,
    NUM_LAYERS_RECOVERY,
    HIDDEN_DIM_DISCRIMINATOR,
    HIDDEN_DIM_GENERATOR,
    WARMUP_EPOCHS,
    LAMBDA_SUP,
    LAMBDA_MOM,
    LAMBDA_SPEC,
    LAMBDA_ADV,
    NUM_CLASSES,
    LABEL_EMB_DIM,
    NOISE_DIM,
    FREQ_DIM,
    FREQ_N_HARMONICS,
    SSVEP_FS,
    ALL_STIM_FREQS,
    LR_CLASSIFIER
)
from src.models.classifier import Classifier
from src.models.recovery import cRecovery
from src.models.generator import cGenerator
from src.training.train_classifier import train_classifier
from src.models.freq_conditioning import build_freq_basis

In [50]:
logger = get_logger("TSTR")

In [51]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

2026-08-06 15:19:38 | INFO | Using device: cuda


In [52]:
test_dataset = EEGH5Dataset("C:\\Users\\danie\\Documents\\Tesis\\Prueba\\BestTimeGAN\\data\\processed\\eeg_test_8.h5", keep_classes=GAN_CLASSES)

In [53]:
test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

logger.info(f"Dataset windows: {len(test_dataset)}")

2026-08-06 15:19:38 | INFO | Dataset windows: 888


In [54]:
if GAN_CLASSES is not None:
    stim_freqs = [ALL_STIM_FREQS[c] for c in GAN_CLASSES]
else:
    stim_freqs = list(ALL_STIM_FREQS)

In [55]:
G = cGenerator(
    z_dim=NOISE_DIM,
    h_dim=HIDDEN_DIM_GENERATOR,
    num_layers=NUM_LAYERS_GENERATOR,
    out_dim=LATENT_DIM,
    freq_dim=FREQ_DIM,
).to(device)


R = cRecovery(
    h_dim=LATENT_DIM,
    x_dim=NUM_CHANNELS,
).to(device)

C = Classifier(9,512,2)

In [56]:
G.load_state_dict(torch.load("C:\\Users\\danie\\Documents\\Tesis\\Prueba\\BestTimeGAN\\checkpoints\\best_timegan_250_psd.pt", weights_only=True)['G'])
R.load_state_dict(torch.load("C:\\Users\\danie\\Documents\\Tesis\\Prueba\\BestTimeGAN\\checkpoints\\recovery_24.pt", weights_only=True))

<All keys matched successfully>

In [57]:
@torch.no_grad()
def generate_eeg(generator, recovery, label, device):
    generator.eval()
    recovery.eval()

    B = BATCH_SIZE
    T = WINDOW_SIZE
    z_dim = NOISE_DIM
    

    z = torch.randn(B, T, z_dim).to(device)
    labels = torch.full((B,), label).to(device)
    freq_basis_local = build_freq_basis(labels, T, stim_freqs, SSVEP_FS, FREQ_N_HARMONICS)
    H_fake = generator(z, freq_basis_local)
    labels_orig = torch.full((B,), GAN_CLASSES[label]).to(device)
    X_fake = recovery(H_fake, labels_orig)

    X_fake = X_fake.cpu().numpy()

    return X_fake

In [75]:
stim_freqs

[16.0, 24.0]

In [73]:
GAN_CLASSES[0]

3

In [74]:
GAN_CLASSES[1]

5

In [58]:
@torch.no_grad()
def generate_dataset(generator, recovery, n_batches, label, device):
    """
    Generate synthetic EEG dataset.

    Returns
    -------
    np.ndarray of shape [N_trials, T, C]
    """
    generator.eval()
    all_samples = []

    for _ in range(n_batches):
        x_fake = generate_eeg(generator, recovery, label, device)
        all_samples.append(x_fake)

    # Concatenate along batch dimension
    data = np.concatenate(all_samples, axis=0)  # [N_total, 512, 9]

    return data

In [59]:
syn_8 = generate_dataset(G, R, 140, 0, device)
syn_16 = generate_dataset(G, R, 140, 1, device)

In [60]:
syn_dataset = EEGClassDataset(
    eeg_list=[syn_8, syn_16],
    labels=[0,1],
    window_size=WINDOW_SIZE,
    hop_size=WINDOW_SIZE,
    normalize=False,
)

In [61]:
train_data, val_data = random_split(syn_dataset, [0.87,0.13])

In [62]:
len(syn_dataset)

8960

In [63]:
train_dataloader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

In [64]:
val_dataloader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

In [65]:
train_classifier(
    C,
    train_dataloader,
    val_dataloader,
    10,
    LR_CLASSIFIER,
    device,
    logger = logger
)

2026-08-06 15:19:42 | INFO | Epoch 1 | Train Loss: 0.0549 | Train Acc: 0.9859 | Val Loss: 0.0002 | Val Acc: 1.0000
2026-08-06 15:19:43 | INFO | Epoch 2 | Train Loss: 0.0005 | Train Acc: 1.0000 | Val Loss: 0.0001 | Val Acc: 1.0000
2026-08-06 15:19:44 | INFO | Epoch 3 | Train Loss: 0.0002 | Train Acc: 1.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
2026-08-06 15:19:45 | INFO | Epoch 4 | Train Loss: 0.0001 | Train Acc: 1.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
2026-08-06 15:19:46 | INFO | Epoch 5 | Train Loss: 0.0001 | Train Acc: 1.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
2026-08-06 15:19:46 | INFO | Epoch 6 | Train Loss: 0.0001 | Train Acc: 1.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
2026-08-06 15:19:47 | INFO | Epoch 7 | Train Loss: 0.0000 | Train Acc: 1.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
2026-08-06 15:19:48 | INFO | Epoch 8 | Train Loss: 0.0000 | Train Acc: 1.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
2026-08-06 15:19:49 | INFO | Epoch 9 | Train Loss: 0.0000 | Train Acc: 1.0000 | 

Classifier(
  (temporal): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
  (spatial): Conv2d(8, 16, kernel_size=(9, 1), stride=(1, 1), groups=8, bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (pool1): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
  (dropout): Dropout(p=0.25, inplace=False)
  (sep): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=16, bias=False)
  (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (pool2): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)

In [66]:
torch.save(C.state_dict(), "TSClassifier3.pt")

In [67]:
C.load_state_dict(torch.load("TSClassifier3.pt", weights_only=True))

<All keys matched successfully>

In [68]:
orig_labels_map = torch.tensor(GAN_CLASSES, dtype=torch.long).to(device) if GAN_CLASSES is not None else None

In [69]:
if orig_labels_map is not None:
    _inv_map = {int(v): i for i, v in enumerate(orig_labels_map.tolist())}
    def _to_local(raw_labels: torch.Tensor) -> torch.Tensor:
        """Remap raw dataset labels to 0-based local indices for Classifier."""
        return torch.tensor(
            [_inv_map[int(l)] for l in raw_labels.tolist()],
            dtype=torch.long,
            device=raw_labels.device,
        )
else:
    def _to_local(raw_labels):
        return raw_labels

In [70]:
def evaluate(model, dataloader, device):
    model.to(device)
    model.eval()  # switch to inference mode


    total = 0
    correct = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():  # disable gradients
        for x, y in dataloader:
            y = _to_local(y)
            x = x.to(device)
            y = y.to(device)

            logits = model(x)              # (B, num_classes)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            all_preds.append(preds.cpu())
            all_labels.append(y.cpu())

    accuracy = correct / total

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    return accuracy, all_preds, all_labels

In [71]:
accuracy, all_preds, all_labels = evaluate(C, test_dataloader, device)

In [72]:
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds))


              precision    recall  f1-score   support

           0       0.49      0.98      0.66       429
           1       0.29      0.01      0.02       435

    accuracy                           0.49       864
   macro avg       0.39      0.49      0.34       864
weighted avg       0.39      0.49      0.33       864

